In [16]:
import yfinance as yf
import numpy as np
import pandas as pd

# Nifty 50 index
nifty = yf.download("^NSEI", start="2015-01-01", end="2024-12-31")

# Individual stocks (sample)
tickers = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS"]
data = yf.download(tickers, start="2015-01-01", end="2024-12-31",
                   group_by='ticker', auto_adjust=True)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  4 of 4 completed


In [17]:
import pandas_ta as ta

df = nifty.copy()
df.columns = df.columns.get_level_values(0).str.lower()

# Trend
df['sma_20']  = ta.sma(df['close'], length=20)
df['ema_9']   = ta.ema(df['close'], length=9)
macd = ta.macd(df['close'])
df = pd.concat([df, macd], axis=1)

# Momentum
df['rsi_14']  = ta.rsi(df['close'], length=14)
df['cci_20']  = ta.cci(df['high'], df['low'], df['close'], length=20)
df['roc_10']  = ta.roc(df['close'], length=10)

# Volatility
bbands = ta.bbands(df['close'], length=20)
df = pd.concat([df, bbands], axis=1)
df['atr_14']  = ta.atr(df['high'], df['low'], df['close'], length=14)

# Volume
df['obv'] = ta.obv(df['close'], df['volume'])

In [18]:
# Daily return
df['return'] = df['close'].pct_change() * 100

# Lagged returns (features)
for lag in [1, 2, 3, 5]:
    df[f'ret_lag_{lag}'] = df['return'].shift(lag)

# Rolling stats
df['roll_std_20'] = df['return'].rolling(20).std()
df['roll_mean_5'] = df['return'].rolling(5).mean()

# --- REGRESSION TARGET ---
df['target_reg'] = df['return'].shift(-1)   # next-day return

# --- CLASSIFICATION TARGET (binary) ---
df['target_cls'] = (df['return'].shift(-1) > 0).astype(int)

# --- CLASSIFICATION TARGET (3-class) ---
def regime(r):
    if   r >  0.5: return 2
    elif r < -0.5: return 0
    else:           return 1

df['target_3cls'] = df['return'].shift(-1).apply(regime)

df.dropna(inplace=True)

In [11]:
# India VIX
vix = yf.download(
    "^INDIAVIX",
    start="2015-01-01",
    end="2024-12-31",
    progress=False
)["Close"]

if hasattr(vix, "columns"):
    vix = vix.iloc[:, 0]

vix.name = "india_vix"

# USD/INR
usdinr = yf.download(
    "USDINR=X",
    start="2015-01-01",
    end="2024-12-31",
    progress=False
)["Close"]

if hasattr(usdinr, "columns"):
    usdinr = usdinr.iloc[:, 0]

usdinr_ret = usdinr.pct_change() * 100
usdinr_ret.name = "usdinr_return"

# Add/overwrite columns directly
df["india_vix"] = vix.reindex(df.index)
df["usdinr_return"] = usdinr_ret.reindex(df.index)

# Forward fill VIX
df["india_vix"] = df["india_vix"].ffill()

In [19]:
# Feature columns (exclude targets)
feat_cols = [c for c in df.columns
             if c not in ['target_reg', 'target_cls', 'target_3cls']]

# Regression dataset
X_reg = df[feat_cols]
y_reg = df['target_reg']

# Classification dataset
X_cls = df[feat_cols]
y_cls = df['target_cls']   # or target_3cls

# Temporal split (no shuffle for time-series!)
split = int(len(df) * 0.8)
X_reg_train, X_reg_test = X_reg.iloc[:split], X_reg.iloc[split:]
y_reg_train, y_reg_test = y_reg.iloc[:split], y_reg.iloc[split:]

X_cls_train, X_cls_test = X_cls.iloc[:split], X_cls.iloc[split:]
y_cls_train, y_cls_test = y_cls.iloc[:split], y_cls.iloc[split:]

# Save
df[[*feat_cols, 'target_reg']].to_csv("nifty_regression.csv")
df[[*feat_cols, 'target_cls']].to_csv("nifty_classification.csv")